# VillageToMarket — Crop Pricing Recommendation Model

This notebook builds an end-to-end **pricing recommendation model** for the VillageToMarket platform.  
Given a new crop listing (structured fields + an attached image), it outputs a **recommended price range** in ZWL per unit.

### Pipeline Overview
```
Crop Listing Fields ─────┐
                          ├─► Unified Feature Vector ─► Regression Model ─► Recommended Price
Listing Image (CNN) ─────┘
```

**Input fields analysed from `Listing` model:**
| Field | Type | Description |
|---|---|---|
| `produce_type` | FK → ProduceType | Crop type (e.g. Tomatoes, Maize) |
| `variety` | CharField | Sub-variety (e.g. Roma, Yellow) |
| `grade` | CharField | Grade A / B / C / Export |
| `is_organic` | BooleanField | Organic certification flag |
| `quality_tags` | JSONField | `['fresh','pesticide-free','locally-grown']` |
| `quantity_available` | DecimalField | Volume available |
| `unit` | CharField | kg / tonnes / bags / crates … |
| `district` | CharField | Growing region |
| `harvest_date` | DateField | Days since harvest (freshness proxy) |
| `available_from/until` | DateField | Listing window |
| `images` (ListingImage) | ImageField | Crop photo for visual quality scoring |

**Target:** `price_per_unit` (ZWL)  
Reference anchor: `MarketPrice.price_avg` per produce_type × district

## Section 1 — Import Required Libraries

In [ ]:
# ── standard data science stack ──────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import date, timedelta
import random, os, io, json

# ── image handling ────────────────────────────────────────────────────────────
try:
    import cv2
    OPENCV_AVAILABLE = True
except ImportError:
    OPENCV_AVAILABLE = False
    print("OpenCV not found — using Pillow fallback for image ops")

from PIL import Image, ImageDraw, ImageFilter
import requests

# ── deep learning (feature extractor) ────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    from tensorflow.keras.preprocessing.image import img_to_array
    TF_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} — GPU: {len(tf.config.list_physical_devices('GPU'))>0}")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not installed — image embedding will use colour-histogram fallback")

# ── ML / modelling ────────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (LabelEncoder, MinMaxScaler,
                                    StandardScaler, OneHotEncoder)
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed — will use GradientBoostingRegressor instead")

print("\nAll imports successful ✓")

## Section 2 — Define Crop Listing Fields Schema

Here we map every field from the backend `Listing` model into the schema our model will consume, validate types, and show a sample row.

In [ ]:
# ── Field schema mirroring the Django Listing model ──────────────────────────

LISTING_SCHEMA = {
    # ── Produce identification ────────────────────────────────────────────────
    "produce_type":        {"dtype": "str",   "source": "FK → ProduceType",  "example": "Tomatoes"},
    "category":            {"dtype": "str",   "source": "FK → Category",     "example": "Vegetables"},
    "variety":             {"dtype": "str",   "source": "CharField",         "example": "Roma"},
    "grade":               {"dtype": "str",   "source": "CharField",         "example": "Grade A"},
    # ── Quality flags ─────────────────────────────────────────────────────────
    "is_organic":          {"dtype": "bool",  "source": "BooleanField",      "example": True},
    "quality_tags":        {"dtype": "list",  "source": "JSONField",         "example": ["fresh","pesticide-free"]},
    # ── Quantity ──────────────────────────────────────────────────────────────
    "quantity_available":  {"dtype": "float", "source": "DecimalField",      "example": 500.0},
    "unit":                {"dtype": "str",   "source": "UNIT_CHOICES",      "example": "kg"},
    "minimum_order":       {"dtype": "float", "source": "DecimalField",      "example": 50.0},
    # ── Freshness / timing ────────────────────────────────────────────────────
    "harvest_date":        {"dtype": "date",  "source": "DateField",         "example": "2026-02-20"},
    "available_from":      {"dtype": "date",  "source": "DateField",         "example": "2026-02-26"},
    "available_until":     {"dtype": "date",  "source": "DateField",         "example": "2026-03-26"},
    # ── Location ──────────────────────────────────────────────────────────────
    "district":            {"dtype": "str",   "source": "CharField",         "example": "Harare"},
    # ── Images ────────────────────────────────────────────────────────────────
    "image_path":          {"dtype": "str",   "source": "ListingImage",      "example": "listings/2026/02/tomatoes.jpg"},
    # ── TARGET ────────────────────────────────────────────────────────────────
    "price_per_unit":      {"dtype": "float", "source": "DecimalField",      "example": 450.0},  # ZWL
}

# Display as table
schema_df = pd.DataFrame(LISTING_SCHEMA).T.reset_index()
schema_df.columns = ["field", "dtype", "source_in_model", "example_value"]
print("=" * 70)
print(" LISTING FIELDS USED BY PRICING MODEL")
print("=" * 70)
print(schema_df.to_string(index=False))

# ── Sample single listing dict ───────────────────────────────────────────────
SAMPLE_LISTING = {k: v["example"] for k, v in LISTING_SCHEMA.items()}
print("\n── Sample listing row ──")
print(json.dumps(
    {k: (str(v) if isinstance(v, (date,)) else v) for k, v in SAMPLE_LISTING.items()},
    indent=2
))

## Section 3 — Load and Preprocess Crop Images

Images are loaded from disk (or generated synthetically), resized to **224×224** for MobileNetV2, normalised, and optionally augmented.

In [ ]:
IMG_SIZE = (224, 224)   # MobileNetV2 expected input
CROP_COLORS = {        # approximate dominant colour per crop type for synthetic images
    "Tomatoes":   (200, 60,  50),
    "Maize":      (240, 210, 60),
    "Cabbage":    (80,  160, 80),
    "Onions":     (200, 160, 80),
    "Potatoes":   (180, 140, 80),
    "Beans":      (160, 100, 60),
    "Groundnuts": (200, 170, 100),
    "Spinach":    (50,  130, 60),
}

def generate_synthetic_crop_image(crop_type: str, quality_score: float) -> np.ndarray:
    """
    Synthesise a simple crop image using PIL.
    Quality score 0-1 controls how 'bright/vivid' vs 'dull/dark' the image looks.
    Returns an RGB numpy array (224, 224, 3).
    """
    base_rgb = CROP_COLORS.get(crop_type, (150, 150, 150))
    brightness = int(quality_score * 60)         # quality → brightness offset
    r = min(255, base_rgb[0] + brightness)
    g = min(255, base_rgb[1] + brightness)
    b = min(255, base_rgb[2] + 20)

    img = Image.new("RGB", IMG_SIZE, color=(r, g, b))
    draw = ImageDraw.Draw(img)
    # Add some texture blobs to mimic produce
    rng = random.Random(hash(crop_type))
    for _ in range(12):
        x = rng.randint(20, 200); y = rng.randint(20, 200)
        radius = rng.randint(15, 40)
        shade = (max(0, r - 30), max(0, g - 30), max(0, b - 20))
        draw.ellipse([x, y, x+radius, y+radius], fill=shade)
    img = img.filter(ImageFilter.GaussianBlur(radius=1))
    return np.array(img)


def load_image_from_path(path: str) -> np.ndarray:
    """Load a real image from disk → RGB (224,224,3) numpy array."""
    if OPENCV_AVAILABLE:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)
    else:
        img_rgb = np.array(Image.open(path).convert("RGB").resize(IMG_SIZE))
    return img_rgb


def augment_image(img: np.ndarray) -> list:
    """Return 4 augmented versions of an image (flip H, flip V, +brightness, -brightness)."""
    augmented = []
    augmented.append(np.fliplr(img))
    augmented.append(np.flipud(img))
    augmented.append(np.clip(img.astype(int) + 25, 0, 255).astype(np.uint8))
    augmented.append(np.clip(img.astype(int) - 25, 0, 255).astype(np.uint8))
    return augmented


# ── Preview synthetic images ──────────────────────────────────────────────────
crops_preview = list(CROP_COLORS.keys())[:4]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, crop in zip(axes, crops_preview):
    img = generate_synthetic_crop_image(crop, quality_score=0.75)
    ax.imshow(img)
    ax.set_title(crop, fontsize=10)
    ax.axis("off")
fig.suptitle("Synthetic Crop Images (quality_score = 0.75)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("Image shape:", img.shape, "| dtype:", img.dtype)

## Section 4 — Extract Image Features Using CNN

We use **MobileNetV2** (pretrained on ImageNet, classification head removed) as a frozen feature extractor.  
Each image → 1280-dim embedding.  If TensorFlow is unavailable, a 48-dim colour-histogram fallback is used.

In [ ]:
IMG_EMBED_DIM = 1280   # MobileNetV2 global avg pool output
HIST_EMBED_DIM = 48   # fallback: 16 bins × 3 channels

# ── Build or load the feature extractor ──────────────────────────────────────
if TF_AVAILABLE:
    base_model = MobileNetV2(weights="imagenet", include_top=False,
                             input_shape=(224, 224, 3), pooling="avg")
    base_model.trainable = False   # freeze — we only use it as extractor
    print(f"MobileNetV2 loaded  — output shape: {base_model.output_shape}")
    EMBED_DIM = IMG_EMBED_DIM
else:
    base_model = None
    EMBED_DIM = HIST_EMBED_DIM
    print(f"Fallback colour-histogram extractor — {EMBED_DIM}-dim")


def extract_image_features(img_array: np.ndarray) -> np.ndarray:
    """
    img_array : RGB numpy array (224, 224, 3) uint8
    Returns   : 1-D float numpy vector
    """
    if TF_AVAILABLE and base_model is not None:
        x = img_to_array(img_array)           # float32
        x = preprocess_input(x)               # MobileNetV2 normalisation
        x = np.expand_dims(x, axis=0)         # (1, 224, 224, 3)
        features = base_model.predict(x, verbose=0)
        return features[0]                    # (1280,)
    else:
        # Colour-histogram fallback (16 bins per channel → 48 values)
        hist_r = np.histogram(img_array[:,:,0], bins=16, range=(0,256))[0]
        hist_g = np.histogram(img_array[:,:,1], bins=16, range=(0,256))[0]
        hist_b = np.histogram(img_array[:,:,2], bins=16, range=(0,256))[0]
        feat = np.concatenate([hist_r, hist_g, hist_b]).astype(np.float32)
        return feat / (feat.sum() + 1e-9)     # L1 normalise


# ── Smoke-test on one synthetic image ─────────────────────────────────────────
test_img  = generate_synthetic_crop_image("Tomatoes", quality_score=0.8)
test_feat = extract_image_features(test_img)
print(f"\nFeature vector shape : {test_feat.shape}")
print(f"Min/max/mean         : {test_feat.min():.4f} / {test_feat.max():.4f} / {test_feat.mean():.4f}")

# Visualise first 60 dimensions
plt.figure(figsize=(12, 2.5))
plt.bar(range(min(60, EMBED_DIM)), test_feat[:60], color="steelblue", width=0.8)
plt.title("CNN Feature Vector — first 60 dimensions (Tomatoes, high quality)")
plt.xlabel("Feature dimension")
plt.ylabel("Activation")
plt.tight_layout()
plt.show()

## Section 5 — Encode Crop Quality Parameters

Categorical fields (`produce_type`, `variety`, `grade`, `unit`, `district`) are **one-hot encoded**.  
Numeric fields (`quantity_available`, `days_since_harvest`, `minimum_order`) are **MinMax scaled** to [0, 1].  
Boolean / tag fields are converted to binary indicators.

In [ ]:
# ── Vocabulary / domain knowledge drawn from Listing model choices ────────────
PRODUCE_TYPES = list(CROP_COLORS.keys())
VARIETIES     = {
    "Tomatoes":   ["Roma", "Cherry", "Beefsteak", "Plum"],
    "Maize":      ["Yellow", "White", "Hybrid", "Open-Pollinated"],
    "Cabbage":    ["Green", "Red", "Savoy", "Napa"],
    "Onions":     ["Red", "White", "Brown", "Spring"],
    "Potatoes":   ["Irish", "Sweet", "Red", "Yellow"],
    "Beans":      ["Sugar", "Kidney", "Black-eyed", "Soya"],
    "Groundnuts": ["Virginia", "Valencia", "Spanish"],
    "Spinach":    ["English", "Baby", "Creole"],
}
GRADES    = ["Grade A", "Grade B", "Grade C", "Export Quality", "Premium", "Standard"]
UNITS     = ["kg", "g", "tonnes", "bags", "crates", "trays", "bundles", "pieces"]
DISTRICTS = ["Harare", "Bulawayo", "Manicaland", "Masvingo",
             "Midlands", "Mashonaland East", "Mashonaland West",
             "Mashonaland Central", "Matabeleland North", "Matabeleland South"]
QUALITY_TAG_LIST = ["fresh", "pesticide-free", "locally-grown", "non-gmo",
                    "sun-dried", "hand-picked", "certified-organic"]


# ── Encoder objects (fit later on full dataset) ───────────────────────────────
_CAT_FIELDS = {
    "produce_type": PRODUCE_TYPES,
    "variety":      [v for vals in VARIETIES.values() for v in vals] + ["Other"],
    "grade":        GRADES,
    "unit":         UNITS,
    "district":     DISTRICTS,
}
le_encoders: dict[str, dict] = {}
for field, vocab in _CAT_FIELDS.items():
    le_encoders[field] = {val: idx for idx, val in enumerate(vocab)}
    le_encoders[field]["__unk__"] = len(vocab)  # unknown token


def encode_tabular_features(row: dict) -> np.ndarray:
    """
    Convert a single listing dict into a fixed-length float vector.
    Total length = sum(len(vocab)+1 for each cat field) + 3 numeric + 1 bool + n tag bits
    """
    parts = []

    # ── one-hot categorical fields ────────────────────────────────────────────
    for field, vocab in _CAT_FIELDS.items():
        vec = np.zeros(len(vocab) + 1, dtype=np.float32)
        val = row.get(field, "__unk__")
        idx = le_encoders[field].get(val, le_encoders[field]["__unk__"])
        vec[idx] = 1.0
        parts.append(vec)

    # ── numeric features (raw — scaler applied at dataset level) ─────────────
    qty  = float(row.get("quantity_available", 0))
    minord = float(row.get("minimum_order", qty * 0.1) or qty * 0.1)

    harvest_str = row.get("harvest_date", str(date.today()))
    if isinstance(harvest_str, date):
        harvest_d = harvest_str
    else:
        harvest_d = date.fromisoformat(str(harvest_str))
    days_since_harvest = max(0, (date.today() - harvest_d).days)

    parts.append(np.array([qty, minord, days_since_harvest], dtype=np.float32))

    # ── boolean ───────────────────────────────────────────────────────────────
    parts.append(np.array([float(row.get("is_organic", False))], dtype=np.float32))

    # ── quality tags multi-hot ────────────────────────────────────────────────
    tags = row.get("quality_tags", [])
    tag_vec = np.array([1.0 if t in tags else 0.0 for t in QUALITY_TAG_LIST], dtype=np.float32)
    parts.append(tag_vec)

    return np.concatenate(parts)


# ── Test encoding on sample listing ──────────────────────────────────────────
sample_row = {
    "produce_type": "Tomatoes", "variety": "Roma", "grade": "Grade A",
    "unit": "kg", "district": "Harare", "is_organic": True,
    "quality_tags": ["fresh", "pesticide-free"], "quantity_available": 500.0,
    "minimum_order": 50.0, "harvest_date": date.today() - timedelta(days=3),
}
tab_vec = encode_tabular_features(sample_row)
TABULAR_DIM = len(tab_vec)
print(f"Tabular feature vector length : {TABULAR_DIM}")
print(f"First 20 values               : {tab_vec[:20]}")

## Section 6 — Build Feature Vector from Fields + Image

Concatenate the encoded tabular features with the CNN image embedding → **unified feature vector** per crop listing.

In [ ]:
def build_full_feature_vector(row: dict, img_array: np.ndarray) -> np.ndarray:
    """Combine tabular + image features into one vector."""
    tab   = encode_tabular_features(row)
    img_f = extract_image_features(img_array)
    return np.concatenate([tab, img_f])


# ── Generate a synthetic dataset of 800 crop listings ────────────────────────
random.seed(42)
np.random.seed(42)

# Price anchors (ZWL per kg-equivalent) per crop — loosely realistic
BASE_PRICES = {
    "Tomatoes": 450, "Maize": 320, "Cabbage": 280,
    "Onions": 390,   "Potatoes": 350, "Beans": 580,
    "Groundnuts": 900, "Spinach": 200,
}
# Grade multipliers
GRADE_MULT = {
    "Export Quality": 1.35, "Premium": 1.25, "Grade A": 1.10,
    "Grade B": 0.95,        "Grade C": 0.80, "Standard": 0.90,
}

N_SAMPLES   = 800
records_tab = []   # tabular dicts
images      = []   # np arrays
prices      = []   # target

print("Generating synthetic dataset …")
for i in range(N_SAMPLES):
    crop      = random.choice(PRODUCE_TYPES)
    variety   = random.choice(VARIETIES.get(crop, ["Standard"]))
    grade     = random.choice(GRADES)
    unit      = random.choice(["kg", "bags", "crates"])
    district  = random.choice(DISTRICTS)
    is_org    = random.random() < 0.25
    qty       = round(random.uniform(20, 2000), 1)
    min_ord   = round(qty * random.uniform(0.05, 0.3), 1)
    days_harv = random.randint(0, 14)
    harvest_d = date.today() - timedelta(days=days_harv)
    tags      = random.sample(QUALITY_TAG_LIST, k=random.randint(0, 3))
    quality_s = random.uniform(0.4, 1.0)   # 0-1 visual quality

    # Compute a ground-truth price with noise
    base   = BASE_PRICES[crop]
    mult   = GRADE_MULT[grade]
    org    = 1.15 if is_org else 1.0
    fresh  = 1.0 + max(0, (7 - days_harv) / 7) * 0.10
    vis    = 0.90 + quality_s * 0.20
    noise  = np.random.normal(1.0, 0.08)
    price  = round(base * mult * org * fresh * vis * noise, 2)

    row = {
        "produce_type": crop, "variety": variety, "grade": grade,
        "unit": unit, "district": district, "is_organic": is_org,
        "quality_tags": tags, "quantity_available": qty,
        "minimum_order": min_ord, "harvest_date": harvest_d,
    }
    records_tab.append(row)
    images.append(generate_synthetic_crop_image(crop, quality_s))
    prices.append(price)

print(f"Dataset ready  → {N_SAMPLES} samples")
print(f"Price range    → ZWL {min(prices):.0f} – {max(prices):.0f}")

# ── Extract ALL features (this is the slow step if TF) ───────────────────────
print("\nExtracting combined feature vectors …")
X_all = np.array([
    build_full_feature_vector(row, img)
    for row, img in zip(records_tab, images)
])
y_all = np.array(prices)

TOTAL_DIM = X_all.shape[1]
print(f"Feature matrix : {X_all.shape}  (tabular {TABULAR_DIM} + image {EMBED_DIM})")

# ── Apply MinMaxScaler to numeric columns in tabular block ────────────────────
# Numeric cols sit at a fixed offset in the tabular block
# We normalise the entire X matrix now so the model sees 0-1 inputs everywhere.
scaler = MinMaxScaler()
X_all  = scaler.fit_transform(X_all)

# DataFrame for quick inspection
df = pd.DataFrame(records_tab)
df["price_per_unit"] = y_all
print("\nSample of dataset:")
print(df[["produce_type","variety","grade","is_organic","district","quantity_available","price_per_unit"]].head(8).to_string(index=False))

## Section 7 — Train Pricing Recommendation Model

We train three candidate regressors on the unified feature vectors:
- **XGBoost** (preferred if installed)
- **GradientBoostingRegressor** (scikit-learn fallback)
- **RandomForestRegressor** (baseline)

80 / 20 train-test split, with 5-fold cross-validation R² reported.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

# ── Candidate models ──────────────────────────────────────────────────────────
models = {}

if XGB_AVAILABLE:
    models["XGBoost"] = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.07,
        subsample=0.85, colsample_bytree=0.8,
        random_state=42, verbosity=0, n_jobs=-1
    )

models["GradientBoosting"] = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.08,
    subsample=0.85, random_state=42
)
models["RandomForest"] = RandomForestRegressor(
    n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
)

# ── Train and cross-validate ──────────────────────────────────────────────────
results = {}
print("\nTraining models …\n" + "-" * 50)
for name, model in models.items():
    model.fit(X_train, y_train)
    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=5, scoring="r2", n_jobs=-1)
    y_pred   = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results[name] = {"model": model, "y_pred": y_pred,
                     "MAE": mae, "RMSE": rmse, "R2": r2,
                     "CV_R2_mean": cv_scores.mean(), "CV_R2_std": cv_scores.std()}
    print(f"  {name:<22}  MAE={mae:6.1f}  RMSE={rmse:6.1f}  R²={r2:.4f}  "
          f"CV-R²={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

# ── Select best model ─────────────────────────────────────────────────────────
best_name  = max(results, key=lambda k: results[k]["R2"])
best_model = results[best_name]["model"]
print(f"\n★  Best model: {best_name}  (R² = {results[best_name]['R2']:.4f})")

## Section 8 — Evaluate Model Performance

Plots: actual vs predicted prices, residuals, and feature importances for the best model.

In [ ]:
best_pred = results[best_name]["y_pred"]

fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Actual vs Predicted ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.scatter(y_test, best_pred, alpha=0.55, color="steelblue", edgecolors="white", s=50)
lims = [min(y_test.min(), best_pred.min()) - 20,
        max(y_test.max(), best_pred.max()) + 20]
ax1.plot(lims, lims, "r--", linewidth=1.5, label="Perfect fit y=x")
ax1.set_xlabel("Actual Price (ZWL / unit)", fontsize=11)
ax1.set_ylabel("Predicted Price (ZWL / unit)", fontsize=11)
ax1.set_title(f"Actual vs Predicted — {best_name}  (R²={results[best_name]['R2']:.4f})", fontsize=12)
ax1.legend()

# ── 2. Residuals histogram ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
residuals = y_test - best_pred
ax2.hist(residuals, bins=30, color="salmon", edgecolor="white", linewidth=0.5)
ax2.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax2.set_xlabel("Residual (Actual − Predicted)", fontsize=10)
ax2.set_ylabel("Count", fontsize=10)
ax2.set_title("Residuals Distribution", fontsize=11)

# ── 3. Metrics comparison bar chart ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
model_names = list(results.keys())
r2_scores   = [results[n]["R2"] for n in model_names]
colors      = ["gold" if n == best_name else "steelblue" for n in model_names]
bars        = ax3.bar(model_names, r2_scores, color=colors, edgecolor="grey")
ax3.set_ylim(0, 1.05)
ax3.set_ylabel("R² Score", fontsize=10)
ax3.set_title("R² Score Comparison", fontsize=11)
for bar, val in zip(bars, r2_scores):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f"{val:.4f}", ha="center", va="bottom", fontsize=9)

# ── 4. MAE & RMSE comparison ──────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
x_pos = np.arange(len(model_names))
mae_vals  = [results[n]["MAE"]  for n in model_names]
rmse_vals = [results[n]["RMSE"] for n in model_names]
ax4.bar(x_pos - 0.2, mae_vals,  0.38, label="MAE",  color="cornflowerblue")
ax4.bar(x_pos + 0.2, rmse_vals, 0.38, label="RMSE", color="coral")
ax4.set_xticks(x_pos); ax4.set_xticklabels(model_names, fontsize=9)
ax4.set_ylabel("ZWL", fontsize=10); ax4.set_title("MAE vs RMSE (lower=better)", fontsize=11)
ax4.legend(fontsize=9)

# ── 5. Feature importance (tabular only — first TABULAR_DIM features) ─────────
ax5 = fig.add_subplot(gs[1, 2])
if hasattr(best_model, "feature_importances_"):
    importance = best_model.feature_importances_[:TABULAR_DIM]
elif hasattr(best_model, "feature_importances_"):
    importance = best_model.feature_importances_[:TABULAR_DIM]
else:
    importance = np.zeros(TABULAR_DIM)

# Build human-readable labels for the first TABULAR_DIM dims
tab_labels = []
for field, vocab in _CAT_FIELDS.items():
    tab_labels += [f"{field}_{v}" for v in vocab] + [f"{field}_unk"]
tab_labels += ["qty", "min_order", "days_since_harvest", "is_organic"]
tab_labels += [f"tag_{t}" for t in QUALITY_TAG_LIST]
tab_labels = tab_labels[:TABULAR_DIM]

# Top-15 tabular features
top_n  = 15
sort_i = np.argsort(importance)[-top_n:]
ax5.barh([tab_labels[i] for i in sort_i], importance[sort_i], color="mediumseagreen")
ax5.set_xlabel("Importance", fontsize=9)
ax5.set_title(f"Top-{top_n} Tabular Features", fontsize=11)

plt.suptitle(f"Pricing Model — Evaluation Dashboard  [{best_name}]",
             fontsize=14, fontweight="bold", y=1.01)
plt.savefig("notebooks/pricing_model_evaluation.png", dpi=120,
            bbox_inches="tight")
plt.show()
print(f"\nSummary for {best_name}:")
print(f"  MAE  = {results[best_name]['MAE']:.2f} ZWL")
print(f"  RMSE = {results[best_name]['RMSE']:.2f} ZWL")
print(f"  R²   = {results[best_name]['R2']:.4f}")

## Section 9 — Predict Price for a New Crop Listing

Pass a **new listing dict + image** through the full pipeline and receive:
- **Recommended price** (point estimate)
- **Price band** (±10 % confidence margin)
- A visual summary card showing the crop image alongside the recommendation

In [ ]:
def recommend_price(
    listing_fields: dict,
    image: np.ndarray | None = None,
    image_path: str | None   = None,
    confidence_margin: float = 0.10,   # ±10%
) -> dict:
    """
    Full pricing recommendation pipeline.

    Parameters
    ----------
    listing_fields    : dict with keys matching LISTING_SCHEMA
    image             : RGB numpy array (224,224,3) — takes priority over image_path
    image_path        : path to image file (used if `image` is None)
    confidence_margin : fractional margin for price band

    Returns
    -------
    dict with keys: recommended_price, price_min, price_max, currency, model_used
    """
    # 1. Resolve image
    if image is None:
        if image_path and os.path.exists(image_path):
            image = load_image_from_path(image_path)
        else:
            # Use a neutral mid-quality synthetic image
            image = generate_synthetic_crop_image(
                listing_fields.get("produce_type", "Tomatoes"),
                quality_score=0.65
            )

    # 2. Build feature vector
    feat_vec = build_full_feature_vector(listing_fields, image)
    feat_vec = scaler.transform(feat_vec.reshape(1, -1))

    # 3. Predict
    raw_price = float(best_model.predict(feat_vec)[0])

    # 4. Confidence band
    margin = raw_price * confidence_margin
    return {
        "recommended_price": round(raw_price, 2),
        "price_min":         round(raw_price - margin, 2),
        "price_max":         round(raw_price + margin, 2),
        "currency":          "ZWL",
        "unit":              listing_fields.get("unit", "kg"),
        "model_used":        best_name,
        "confidence_margin": f"±{int(confidence_margin*100)}%",
    }


# ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
#  DEMO: new listing coming from the app
# ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
new_listing = {
    "produce_type":       "Tomatoes",
    "variety":            "Roma",
    "grade":              "Grade A",
    "unit":               "kg",
    "district":           "Harare",
    "is_organic":         True,
    "quality_tags":       ["fresh", "pesticide-free", "locally-grown"],
    "quantity_available": 300.0,
    "minimum_order":      30.0,
    "harvest_date":       date.today() - timedelta(days=2),
}

# Simulate farmer attaching a photo (synthetic here)
demo_image = generate_synthetic_crop_image("Tomatoes", quality_score=0.88)

result = recommend_price(new_listing, image=demo_image)

# ── Visual summary card ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                         gridspec_kw={"width_ratios": [1, 2]})

# Left: crop image
axes[0].imshow(demo_image)
axes[0].set_title(f"{new_listing['produce_type']} — {new_listing['variety']}", fontsize=12)
axes[0].axis("off")

# Right: recommendation card
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.axis("off")

card_data = [
    ("Crop",            f"{new_listing['produce_type']} ({new_listing['variety']})"),
    ("Grade",           new_listing["grade"]),
    ("Organic",         "Yes" if new_listing["is_organic"] else "No"),
    ("Quality Tags",    ", ".join(new_listing["quality_tags"])),
    ("Quantity",        f"{new_listing['quantity_available']} {new_listing['unit']}"),
    ("District",        new_listing["district"]),
    ("Days since harvest", str((date.today() - new_listing["harvest_date"]).days)),
    ("",                ""),
    ("💰 Recommended Price", f"ZWL {result['recommended_price']:.2f} / {result['unit']}"),
    ("📊 Price Band",   f"ZWL {result['price_min']:.2f}  –  {result['price_max']:.2f}"),
    ("Confidence",      result["confidence_margin"]),
    ("Model",           result["model_used"]),
]

y_pos = 9.3
for label, value in card_data:
    if label == "":
        y_pos -= 0.3; continue
    weight = "bold" if "💰" in label or "📊" in label else "normal"
    fsize  = 12 if weight == "bold" else 10
    color  = "darkgreen" if "💰" in label else ("steelblue" if "📊" in label else "black")
    ax.text(0.3, y_pos, f"{label}:", fontsize=fsize, va="top",
            color="grey", fontweight="normal")
    ax.text(4.8, y_pos, value, fontsize=fsize, va="top",
            color=color, fontweight=weight)
    y_pos -= 0.75

ax.add_patch(plt.Rectangle((0.1, 0.1), 9.8, 9.7,
             linewidth=2, edgecolor="green", facecolor="#f7fff7", zorder=0))
axes[1].set_title("Pricing Recommendation Card", fontsize=13, fontweight="bold", pad=10)

plt.tight_layout()
plt.savefig("notebooks/pricing_recommendation_card.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n" + "="*55)
print("  PRICING RECOMMENDATION OUTPUT (JSON)")
print("="*55)
print(json.dumps(result, indent=2))

---
## Section 10 — Save Model & Integration Notes

The trained model + scaler are serialised with `joblib` so the FastAPI pricing endpoint (`/api/v1/pricing/recommend`) can load and call `recommend_price()` at runtime.

In [ ]:
import joblib, pathlib

MODELS_DIR = pathlib.Path("notebooks/saved_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the best regression model
model_path  = MODELS_DIR / "pricing_model.joblib"
scaler_path = MODELS_DIR / "feature_scaler.joblib"

joblib.dump(best_model, model_path)
joblib.dump(scaler,     scaler_path)

print(f"Model  saved → {model_path}")
print(f"Scaler saved → {scaler_path}")

# ── Integration snippet for api/v1/endpoints/pricing.py ──────────────────────
snippet = '''
# --- Add to api/v1/endpoints/pricing.py ---

import joblib, numpy as np
from notebooks.pricing_recommendation_model import (
    encode_tabular_features, extract_image_features, scaler
)

_MODEL  = joblib.load("notebooks/saved_models/pricing_model.joblib")
_SCALER = joblib.load("notebooks/saved_models/feature_scaler.joblib")

@router.post("/recommend")
async def recommend_price_endpoint(
    produce_type: str,
    variety: str,
    grade: str,
    unit: str,
    district: str,
    is_organic: bool,
    quality_tags: list[str],
    quantity_available: float,
    minimum_order: float,
    harvest_date: str,
    image: UploadFile = File(None),
):
    from datetime import date
    row = {
        "produce_type": produce_type, "variety": variety,
        "grade": grade, "unit": unit, "district": district,
        "is_organic": is_organic, "quality_tags": quality_tags,
        "quantity_available": quantity_available,
        "minimum_order": minimum_order,
        "harvest_date": date.fromisoformat(harvest_date),
    }
    if image:
        contents = await image.read()
        img_arr  = np.array(Image.open(io.BytesIO(contents)).convert("RGB").resize((224,224)))
    else:
        img_arr = None

    return recommend_price(row, image=img_arr)
'''
print("\n" + "─"*60)
print("Integration snippet for pricing.py endpoint:")
print("─"*60)
print(snippet)

print("\n✓ Notebook complete. All artefacts saved in notebooks/saved_models/")